Reference: https://j2rooong.tistory.com/entry/Pytorch-Transformer-Architecture-구현하기

In [ ]:
import torch
import torch.nn as nn
import math

# ----------------------------------------------------------------
# 1. Input Embedding
# ----------------------------------------------------------------
class InputEmbeddings(nn.Module):
    """
    create an input embedding
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        # table that should be trained, should be updated 
        self.embedding = nn.Embedding(vocab_size, d_model)  

    def forward(self, x):
        # matrix * constant
        # To prevent a positional embedding from shadowing the input embedding
        return self.embedding(x) * math.sqrt(self.d_model)
    
# ----------------------------------------------------------------
# 2. Positional Encoding
# ----------------------------------------------------------------
class PositionalEncoding(nn.Module):
    # This method doesnt return anything (None)
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None: 
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        
        # 2D tensor with 0s
        pe = torch.zero(seq_len, d_model)   
        # 2D tensor (seq_len, 1)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(dim=1)

        _2i = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float))

        # Giving each word an unique barcode
        # A clock with 512 needles
        # 0::2 -> from 0 to the end, with a step of 2
        pe[:, 0::2] = torch.sin(position/10000**(_2i/d_model))
        pe[:, 1::2] = torch.cos(position/10000**(_2i/d_model))

        pe = pe.unsqueeze(dim=0)  # 3D tensor (1, seq_len, d_model)
        # While sending data, a buffer keeps the data temporarily
        # With this module, the buffer is saved and loaded when the model is.
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Input x + positional encoding
        # No need to back-propagate
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        # To avoid overfitting, change some of elements into O
        return self.dropout(x)

# ----------------------------------------------------------------
# 3. Layer Nomalization
# ----------------------------------------------------------------
class LayerNormalization(nn.Module):
    def __init__(self, eps: float = 10**-6) -> None:
        super.__init__()
        self.eps = eps
        # multiplied
        # [1.0]
        #  ex) torch.ones(3) --> [1.0 1.0 1.0]
        self.alpha = nn.Parameter(torch.ones(1))
        # Addded
        # [0.0]
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x_size = (seq_len, d_model)
        # the mean of values in the last dimension
        mean = x.mean(dim=-1, keepdim=True) # (seq_len, 1)
        std = x.std(dim=-1, keepdim=True)   # (seq_len, 1)
        # (x - mean) => broadcasting
        return self.alpha * (x-mean) / (std + self.eps) + self.bias


# ----------------------------------------------------------------
# 4. Feed Forward
# ----------------------------------------------------------------
class FeedForwardBlock(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))
    

# ----------------------------------------------------------------
# 5. Multi-Head Attention
# ----------------------------------------------------------------
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super.__init__()
        self.d_model = d_model
        # the number of heads
        # ex) 8
        self.h = h
        # AssertionError
        # When not meeting the condition , it stops
        assert d_model % h == 0, 'd_model is not divisible by h'

        # the number of dimensions that each head handles.
        # ex) 512 / 8 = 64
        self.d_k = d_model // h

        # Linear layers for making Q, K, V
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        # A linear layer to sum up the results from the heads
        self.w_o = nn.Linear(d_model, d_model)
        # to avoid overfitting
        self.dropout = nn.Dropout(dropout)


    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]   # (batch, seq_len, d_model)
        # @: matrix multiplication
        # .transpose(-2, -1): swap the second-to-last dimension with the last dimension
        # for scaling
        # How relevant between two tokens
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k) #(batch, h, seq_len, seq_len)

        # If mask is specified
        if mask is not None:
            # ex) mask = -1e9, then fill -1e9 instead of 0
            attention_scores.masked_fill_(mask==0, -1e9)

        # attention scores -> probability distribution (0~1)
        attention_scores = attention_scores.softmax(dim=-1)  # (batch, seq_len, seq_len)
        # If dropout is provided
        if dropout is not None:
            # apply dropout to attention_scores
            attention_scores = dropout(attention_scores)
        return (attention_scores @ value), attention_scores
    
    def forward(self, q, k, v, mask):
        # 1. Q, K, V -> project into d_k, d_k, d_v dimension
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        # 2. Q, K, V separate into #heads
        # (batch, seq_len, d_model) -> (batch, seq_len, h, d_k) -> (batch, h, seq_len, d_k)
        # in order for each head to compute independently and paralle
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # 3. the actual attention scores
        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # 4. sum all the heads
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)
        
        # 5. final output
        return self.w_o(x) 

In [ ]:
# ----------------------------------------------------------------
# 6. Residual Connection
# ----------------------------------------------------------------
class ResidualConnection(nn.Module):
    
    def __init__(self, dropout : float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization()
        
    def forward(self,x,sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

In [ ]:
# ----------------------------------------------------------------
# 7. EncoderBlock
# ----------------------------------------------------------------
class EncoderBlock(nn.Module):
    
    def __init__(self, self_attention_block: MultiHeadAttentionBlock, feed_forward_block : FeedForwardBlock, dropout : float) -> None:
        super().__init__()
        self.self_atttention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(2)])
        
    def forward(self,x,src_mask):
        #attention: skip connection
        x = self.residual_connections[0](x, lambda x: self.self_atttention_block(x,x,x,src_mask))
        #feed forward: skip connection
        x = self.residual_connections[1](x, self.feed_forward_block)
        return x
# ----------------------------------------------------------------
# 8. Encoder
# ----------------------------------------------------------------
class Encoder(nn.Module):
    
    def __init__(self, layers : nn.ModuleList) -> None : 
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization()
        
    def forward(self,x,mask):
        for layer in self.layers:
            x = layer(x,mask)
        return self.norm(x)

In [ ]:
# ----------------------------------------------------------------
# 9. DecoderBlock
# ----------------------------------------------------------------
class DecoderBlock(nn.Module):
    
    def __init__(self,self_attention_block : MultiHeadAttentionBlock, cross_attention_block : MultiHeadAttentionBlock, feed_forward_block : FeedForwardBlock,dropout:float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.Module([ResidualConnection(dropout) for _ in range(3)])
    
    #tgt_mask:  디코더의 현재 위치 이후의 단어들을 가려주는 마스크 
    #src_mask:  인코더 출력에서 패딩 토큰에 해당하는 위치를 0으로, 실제 단어에 해당하는 위치를 1로 채운 이진 마스크
    #단어 길이를 맞추기 위함(연산량 감소)
    def forward(self,x,encoder_output, src_mask, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x,x,x,tgt_mask))
        #encoder output 값을 받는다 (cross_attention)
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x,encoder_output,encoder_output,src_mask))
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x

# ----------------------------------------------------------------
# 10. Decoder
# ----------------------------------------------------------------
class Decoder(nn.Module):
    
    def __init__(self,layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm  = LayerNormalization()
        
    def forward(self,x,encoder_output,src_mask,tgt_mask):
        for layer in self.layers:
            x = layer(x,encoder_output,src_mask,tgt_mask)
        return self.norm(x)

In [ ]:
# ----------------------------------------------------------------
# 11. Projection Layer
# ----------------------------------------------------------------
class ProjectionLayer(nn.Module):
    
    def __init__(self,d_model : int, vocab_size : int) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)
        
    def forward(self,x):
        #(Batch, seq_len,d_model) -> (Batch,seq_len,vocab_size)
        return torch.log_softmax(self.proj(x), dim=-1)

In [ ]:
# ----------------------------------------------------------------
# 12. Transformer
# ----------------------------------------------------------------
class Transformer(nn.Module):
    
    def __init__(self,encoder :Encoder, decoder : Decoder, src_embed : InputEmbeddings, tgt_embed : InputEmbeddings, src_pos : PositionalEncoding, tgt_pos :PositionalEncoding, projection_layer : ProjectionLayer) -> None: 
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer
        
        
    def encode(self, src,src_mask):
        src = self.src_embed(src)
        src = self.src_pos(src)
        # Input embedding + positional encoding
        return self.encoder(src,src_mask)
    
    def decode(self,encoder_output,src_mask,tgt,tgt_mask):
        tgt = self.tgt_embed(tgt)
        tgt = self.tgt_pos(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)
    
    def project(self,x):
        return self.projection_layer(x)
    
def build_transformer(src_vocab_size : int, tgt_vocab_size : int, src_seq_len : int, tgt_seq_len : int, d_model : int=512, N:int = 6, h : int = 8, dropout : float=0.1, d_ff : int=2048 ) -> Transformer:
    #Create Embedding layers
    src_embed = InputEmbeddings(d_model, src_vocab_size)   
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)
    
    # Create positional encoding layers
    # seq_len: maximum num of words, so that the model can create positional encoding in advance
    src_pos = PositionalEncoding(d_model,src_seq_len,dropout)
    tgt_pos = PositionalEncoding(d_model,tgt_vocab_size,dropout)
    
    #Create encoder blocks
    encoder_blocks=[]
    for _ in range(N):
        encoder_self_attention_block = MultiHeadAttentionBlock(d_model,h,dropout)
        feed_forward_block = FeedForwardBlock(d_model,d_ff, dropout)
        encoder_block = EncoderBlock(encoder_self_attention_block, feed_forward_block, dropout)
        encoder_blocks.append(encoder_block)
        
    #Create decoder blocks
    decoder_blocks=[]
    for _ in range(N):
        decoder_self_attention_block = MultiHeadAttentionBlock(d_model,h,dropout)
        decoder_cross_attention_block = MultiHeadAttentionBlock(d_model,h,dropout)
        feed_forward_block = FeedForwardBlock(d_model,d_ff, dropout)
        decoder_block = DecoderBlock(decoder_self_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
        decoder_blocks.append(decoder_block)

    #Create encoder and decoder
    encoder = Encoder(nn.ModuleList(encoder_blocks))
    decoder = Decoder(nn.ModuleList(decoder_blocks))
    
    #Create projection layer
    projection_layer = ProjectionLayer(d_model, tgt_vocab_size)
    
    #Create the transformer
    transformer = Transformer(encoder, decoder, src_embed,tgt_embed, src_pos, tgt_pos, projection_layer)
    
    #initial parameters
    for p in transformer.parameters():
        if p.dim() >1 : 
            nn.init.xavier_uniform_(p)
            
    return transformer

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


# ----------------------------------------------------------------
# 13. Training
# ----------------------------------------------------------------


def train_model(model: Transformer, train_dataloader: DataLoader, optimizer: torch.optim.Optimizer, criterion: nn.Module, device: torch.device):
    model.train() # training mode (dropout 활성화)
    
    pad_idx = 0 
    
    for batch_idx, batch in enumerate(train_dataloader):
        # shape: (batch_size, seq_len)
        src = batch['src_ids'].to(device) 
        tgt = batch['tgt_ids'].to(device)
        
        # 1. decoder input
        # 인풋 문장 예시:  [SOS] I am a student [EOS]
        # tgt_input:       [SOS] I am a student         <-- 마지막 EOS 제거 (디코더 입력용)
        # tgt_label:             I am a student [EOS]   <-- 첫 SOS 제거 (예측해야 할 정답)
        tgt_input = tgt[:, :-1]
        tgt_label = tgt[:, 1:]
        
        # 2. mask
        src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2) 
        tgt_padding_mask = (tgt_input != pad_idx).unsqueeze(1).unsqueeze(2)
        
        sz = tgt_input.size(1)
        causal_mask = torch.triu(torch.ones(1, 1, sz, sz, device=device), diagonal=1) == 0
        # padding mask + casual mask
        tgt_mask = tgt_padding_mask & causal_mask
        
        # 3. forward Pass
        optimizer.zero_grad() # 이전 그레디언트 초기화
        
        encoder_output = model.encode(src, src_mask)
        decoder_output = model.decode(encoder_output, src_mask, tgt_input, tgt_mask)
        logits = model.project(decoder_output) # shape: (batch_size, tgt_seq_len-1, tgt_vocab_size)
        
        # 4. loss & Backpropagation
        # 크로스 엔트로피에 넣기 위해 행렬 모양을 (N, 클래스수) 형태로 펼쳐
        loss = criterion(
            logits.view(-1, logits.shape[-1]), 
            tgt_label.view(-1)
        )
        
        loss.backward()
        
        # 5. weight Update)
        optimizer.step()
        
        if batch_idx % 100 == 0:
            print(f"Batch {batch_idx} | Loss: {loss.item():.4f}")